# Day 4 — Clean ML Workflow

**Lesson focus:** splits, leakage, baselines, overfitting, and pipelines.

**Required evidence:** Build a clean train/validation/test pipeline.

We will use a small binary text-classification example: `0 = factual` and `1 = clickbait`. The goal today is the **workflow**, not achieving a high score.

## 1. Imports

In [1]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

## 2. Create the practice dataset

A real project would load a cleaned dataset from disk. For Day 4, this small balanced dataset keeps the focus on the ML workflow.

In [2]:
factual_texts = [
    'Government publishes annual budget report',
    'Election commission announces voting schedule',
    'University releases new admission guidelines',
    'Football team wins league match two one',
    'Central bank keeps interest rate unchanged',
    'City council approves road repair project',
    'Hospital opens new emergency care unit',
    'Weather office forecasts rain this weekend',
    'Company reports quarterly revenue growth',
    'School board changes examination schedule',
    'Researchers publish results of health study',
    'Court announces decision in public case',
    'National team names squad for tournament',
    'Airport adds two international flight routes',
    'Ministry releases updated education policy',
    'Local market opens at eight tomorrow',
    'Police issue traffic advisory for festival',
    'Railway announces revised ticket prices',
    'Museum opens exhibition on regional history',
    'Scientists record seasonal river level changes',
]

clickbait_texts = [
    'You will not believe what happened next',
    'This shocking secret has everyone talking',
    'What this celebrity did next stunned fans',
    'The truth they never wanted you to know',
    'Unbelievable moment caught everyone by surprise',
    'This one trick will change your life',
    'Everyone is talking about this shocking reveal',
    'You need to see what happened next',
    'The hidden reason behind this unbelievable story',
    'Fans cannot believe this surprise announcement',
    'What happened next left viewers speechless',
    'This secret detail changes everything',
    'Nobody expected this shocking turn of events',
    'Wait until you see the final result',
    'The real story will completely surprise you',
    'One surprising fact everyone needs to know',
    'This incredible discovery has people amazed',
    'Can you guess what happened after this',
    'The answer will leave you completely shocked',
    'What they found next was truly unbelievable',
]

df = pd.DataFrame({
    'text': factual_texts + clickbait_texts,
    'label': [0] * len(factual_texts) + [1] * len(clickbait_texts),
})

print('Shape:', df.shape)
print('Label counts:')
print(df['label'].value_counts().sort_index())
df.head()

Shape: (40, 2)
Label counts:
label
0    20
1    20
Name: count, dtype: int64


,text,label
0,Government publishes annual budget report,0
1,Election commission announces voting schedule,0
2,University releases new admission guidelines,0
3,Football team wins league match two one,0
4,Central bank keeps interest rate unchanged,0


## 3. Train / validation / test split

We use approximately **70% train, 15% validation, 15% test**.

- **Train:** learn model parameters and vocabulary.
- **Validation:** compare choices and inspect generalization while developing.
- **Test:** use only after the workflow/model choice is finalized.

`stratify` preserves the class balance in each split. A fixed random state makes the split reproducible.

In [3]:
X = df['text']
y = df['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(X_train), len(X_val), len(X_test)],
    'positive_rate': [y_train.mean(), y_val.mean(), y_test.mean()],
})
split_summary

,split,rows,positive_rate
0,train,28,0.5
1,validation,6,0.5
2,test,6,0.5


## 4. Leakage: what must not happen

**Data leakage** happens when information from validation/test data influences training. For text ML, a common mistake is fitting TF-IDF on the full dataset before splitting.

Bad pattern:

```python
X_all = TfidfVectorizer().fit_transform(df['text'])  # fitted before split
```

Our pipeline avoids this: `fit(X_train, y_train)` fits both TF-IDF vocabulary/statistics and the classifier using **training data only**.

## 5. Establish a baseline

A baseline answers: *Is our real model doing better than a trivial strategy?* `DummyClassifier(strategy='most_frequent')` always predicts the most common training class.

In [4]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train.to_frame(), y_train)
baseline_val_pred = baseline.predict(X_val.to_frame())
baseline_val_accuracy = accuracy_score(y_val, baseline_val_pred)

print(f'Baseline validation accuracy: {baseline_val_accuracy:.3f}')

Baseline validation accuracy: 0.500


## 6. Build a leakage-safe ML pipeline

The pipeline connects **TF-IDF vectorization → Logistic Regression**. Calling `.fit()` once on the training text keeps preprocessing and model training together.

In [5]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2))),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

model.fit(X_train, y_train)

print('Training rows used by model:', len(X_train))
print('TF-IDF features learned from training data:',
      len(model.named_steps['tfidf'].vocabulary_))

Training rows used by model: 28
TF-IDF features learned from training data: 278


## 7. Validation and overfitting check

Overfitting means a model learns the training data very well but generalizes poorly. A simple warning sign is a large train-versus-validation performance gap. This gap is a diagnostic, not a complete proof of overfitting.

In [6]:
train_pred = model.predict(X_train)
val_pred = model.predict(X_val)

train_accuracy = accuracy_score(y_train, train_pred)
val_accuracy = accuracy_score(y_val, val_pred)
generalization_gap = train_accuracy - val_accuracy

print(f'Train accuracy:      {train_accuracy:.3f}')
print(f'Validation accuracy: {val_accuracy:.3f}')
print(f'Generalization gap:  {generalization_gap:.3f}')
print(f'Improvement over baseline: {val_accuracy - baseline_val_accuracy:+.3f}')

print('\nValidation classification report:')
print(classification_report(y_val, val_pred, digits=3, zero_division=0))

Train accuracy:      1.000
Validation accuracy: 0.833
Generalization gap:  0.167
Improvement over baseline: +0.333

Validation classification report:
              precision    recall  f1-score   support

           0      1.000     0.667     0.800         3
           1      0.750     1.000     0.857         3

    accuracy                          0.833         6
   macro avg      0.875     0.833     0.829         6
weighted avg      0.875     0.833     0.829         6



## 8. Final test evaluation

Only after the workflow has been chosen do we evaluate the untouched test set. Repeatedly checking the test set while tuning would make it behave like another validation set.

In [7]:
test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_pred)

print(f'Final test accuracy: {test_accuracy:.3f}')
print('\nTest classification report:')
print(classification_report(y_test, test_pred, digits=3, zero_division=0))

Final test accuracy: 0.833

Test classification report:
              precision    recall  f1-score   support

           0      1.000     0.667     0.800         3
           1      0.750     1.000     0.857         3

    accuracy                          0.833         6
   macro avg      0.875     0.833     0.829         6
weighted avg      0.875     0.833     0.829         6



## 9. Required evidence — clean pipeline summary

In [8]:
workflow_report = pd.DataFrame({
    'item': [
        'Train rows', 'Validation rows', 'Test rows',
        'Baseline validation accuracy', 'Model train accuracy',
        'Model validation accuracy', 'Generalization gap',
        'Final test accuracy', 'Leakage-safe preprocessing'
    ],
    'value': [
        len(X_train), len(X_val), len(X_test),
        round(baseline_val_accuracy, 3), round(train_accuracy, 3),
        round(val_accuracy, 3), round(generalization_gap, 3),
        round(test_accuracy, 3), 'Yes — TF-IDF fitted inside training pipeline'
    ],
})
workflow_report

,item,value
0,Train rows,28
1,Validation rows,6
2,Test rows,6
3,Baseline validation accuracy,0.5
4,Model train accuracy,1.0
5,Model validation accuracy,0.833
6,Generalization gap,0.167
7,Final test accuracy,0.833
8,Leakage-safe preprocessing,Yes — TF-IDF fitted inside training pipeline


## 10. Verification

These assertions confirm the split is complete, non-overlapping by row index, stratified for this example, and that the fitted pipeline can make predictions.

In [9]:
assert len(X_train) + len(X_val) + len(X_test) == len(df)
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
assert set(y_train.unique()) == {0, 1}
assert set(y_val.unique()) == {0, 1}
assert set(y_test.unique()) == {0, 1}
assert len(model.predict(['You will not believe this shocking secret'])) == 1

print('All Day 4 ML workflow checks passed successfully!')

All Day 4 ML workflow checks passed successfully!


## Independent practice

1. Change the split to 80/10/10 while keeping it stratified.
2. Replace `most_frequent` with another `DummyClassifier` strategy and compare results.
3. Change TF-IDF from `(1, 2)` to `(1, 1)` and compare validation performance.
4. Explain in your own words why fitting TF-IDF before the split is leakage.
5. Explain why the test set should not be checked after every model change.